# 03. What the deadline did

**COMM 1280 Final Data Project** · Jesus Aguirre and Lisandro Collazo

## The question

Philadelphia used to forgive all property tax on a new home for ten years. The
city cut that benefit and added a 1% tax on home construction. Both started on
1 January 2022. To keep the old deal, a builder had to **have a building permit
in hand by 31 December 2021**.

That detail helps us. The rule was written around the permit itself, so
counting permits measures exactly what the law was aimed at.

We ask two things:

1. **Timing.** Did builders rush to get permits before the deadline?
2. **Level.** Afterwards, did building go back to its old pace or stay lower?

The second question is the important one. A rush by itself only shows that
projects moved around.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

RAW = '../data/raw/'
CLEAN = '../data/cleaned/'
NAVY = '#0B2545'
GOLD = '#C9A227'

os.makedirs(CLEAN, exist_ok=True)

permits = pd.read_csv(RAW + 'new_construction_building_permits_2016_2024.csv')
print('permits loaded:', len(permits))
permits.head()

## Cleaning the data

We need the number of permits per month.

The date column looks like `2021-12-15 14:30:00+00`. The first seven
characters are the year and the month, so we can just take those.

In [ ]:
permits['month'] = permits['permitissuedate'].str[:7]

permits[['permitissuedate', 'month']].head()

In [ ]:
# Split the permits into homes and commercial buildings.
homes = permits[permits['permitdescription'] == 'Residential Building Permit']
commercial = permits[permits['permitdescription'] == 'Commercial Building Permit']

print('home permits:', len(homes))
print('commercial permits:', len(commercial))

In [ ]:
# Count how many of each were issued in every month.
homes_per_month = homes.groupby('month').size()
commercial_per_month = commercial.groupby('month').size()

monthly = pd.DataFrame()
monthly['residential'] = homes_per_month
monthly['commercial'] = commercial_per_month
monthly = monthly.fillna(0)
monthly = monthly.reset_index()

# Notebook 02 showed the data is only comparable from April 2020 on.
monthly = monthly[monthly['month'] >= '2020-04']

monthly.to_csv(CLEAN + 'monthly_new_construction_2020_2024.csv', index=False)
print('months of data:', len(monthly))
monthly.head()

## The picture

One bar for each month. The deadline month is coloured gold.

In [ ]:
# Colour every bar navy, except December 2021.
colours = []
for month in monthly['month']:
    if month == '2021-12':
        colours.append(GOLD)
    else:
        colours.append(NAVY)

plt.figure(figsize=(12, 4))
plt.bar(monthly['month'], monthly['residential'], color=colours)
plt.xticks(rotation=90)
plt.ylabel('Permits issued')
plt.title('New home construction permits issued each month')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Question one: did they rush?

We compare December 2021 against a normal month. For a normal month we use
the average of January through October 2021, which is before the rush started.

In [ ]:
# The ten months before the rush.
normal_months = monthly[(monthly['month'] >= '2021-01') & (monthly['month'] <= '2021-10')]
normal_average = normal_months['residential'].mean()

# Pull out single months we want to talk about.
november = monthly[monthly['month'] == '2021-11']['residential'].sum()
december = monthly[monthly['month'] == '2021-12']['residential'].sum()
january = monthly[monthly['month'] == '2022-01']['residential'].sum()
last_december = monthly[monthly['month'] == '2020-12']['residential'].sum()

print('Normal month in 2021:', round(normal_average))
print('November 2021:', november)
print('December 2021:', december)
print('January 2022:', january)
print('December 2020, for comparison:', last_december)
print()
print('December 2021 was', round(december / normal_average, 1), 'times a normal month.')

Yes. A normal month in 2021 was about 98 permits. December 2021 was **642**.
The next month was 36.

December 2020 was 201, so this is not just an end of year pattern. It is the
deadline.

## Question two: moved, or lost?

If builders only moved their plans earlier, the rush and the quiet afterwards
should cancel each other out.

So we count the extra permits squeezed into the rush, then count how far below
normal the next three years ran. If the two match, building was only moved. If
the gap is bigger, building was lost.

In [ ]:
# What a normal year looked like before the deadline.
normal_year = normal_average * 12

# Extra permits in the two rush months, above what two normal months would be.
pulled_forward = november + december - (2 * normal_average)

# What actually happened in the three years after.
after = monthly[monthly['month'] >= '2022-01']
after_total = after['residential'].sum()
after_per_year = after_total / 3

# How far short of normal those three years fell.
shortfall = (3 * normal_year) - after_total

print('Normal pace before the deadline, per year:', round(normal_year))
print('Extra permits pulled into Nov and Dec 2021:', round(pulled_forward))
print('Actual pace 2022 to 2024, per year:', round(after_per_year))
print('Total shortfall since:', round(shortfall))
print()
print('The shortfall is', round(shortfall / pulled_forward, 1), 'times the size of the rush.')

The hole left behind is about two and a half times bigger than the rush that
came before it. So the deadline did not simply move building earlier. Permits
dropped to about half the old pace and stayed there.

**We cannot call this proof.** Interest rates went up sharply in 2022, and
that would slow down building on its own. Separating the two is the main job
left.

## Our comparison group did not work

To check that this was not just a general slowdown, we planned to compare
homes against commercial buildings, which we thought the law left alone.

It did not. A second city ordinance cut the commercial tax break on the same
day. Here is what that looks like.

In [ ]:
commercial_normal = normal_months['commercial'].mean()
commercial_december = monthly[monthly['month'] == '2021-12']['commercial'].sum()
commercial_after = after['commercial'].mean()
after_average = after['residential'].mean()

print('HOMES')
print('  normal month before:', round(normal_average, 1))
print('  average month after:', round(after_average, 1))
print('  December 2021 spike:', round(december / normal_average, 1), 'times normal')
print()
print('COMMERCIAL')
print('  normal month before:', round(commercial_normal, 1))
print('  average month after:', round(commercial_after, 1))
print('  December 2021 spike:', round(commercial_december / commercial_normal, 1), 'times normal')

Commercial permits spiked 5.6 times in the very same month, and dropped by a
similar amount afterwards. The two groups moved together, so commercial cannot
be our comparison group.

Finding this is a result, not a mistake. It tells us we need a group that
neither law touched.

## Summary

| What we found | The evidence |
| --- | --- |
| Builders rushed the deadline | 642 permits in December 2021, against 98 in a normal month |
| The rush stopped instantly | 36 permits in January 2022 |
| Building did not recover | 628 a year since, against 1,177 before |
| The hole is bigger than the rush | 1,649 short, against 664 pulled forward |
| Commercial is not a usable comparison | It lost its own tax break the same day |

### Things to keep in mind

1. A permit is permission to build, not a finished building. Some are never
   built, and a builder could have taken one in December 2021 just to lock in
   the tax break.
2. Interest rates rose steeply in 2022 and could explain part of the drop.
3. Our data starts in April 2020, so part of the before period is during the
   pandemic.

Next: `04_assessments_and_neighborhoods.ipynb`.